# Agri-Master: Fine-tune YOLOv9 on pest detection dataset

Run this in Google Colab (free GPU). Before running:

1. **Runtime -> Change runtime type -> T4 GPU**, then Save.
2. Click the key icon (Secrets) in the left sidebar. Add a new secret named `ROBOFLOW_API_KEY` with your Roboflow private API key as the value, and enable notebook access.
   - We use Colab Secrets instead of typing the key directly in a cell, because this notebook gets pushed to a public GitHub repo, and a hardcoded key would be exposed to anyone who views the file.

In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

## Load API key from Secrets (falls back to manual entry if not using Colab)

In [ ]:
ROBOFLOW_API_KEY = None
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    pass

if not ROBOFLOW_API_KEY:
    from getpass import getpass
    ROBOFLOW_API_KEY = getpass("Enter your Roboflow API key: ")

## Download the raw 97-class dataset (same source as our local copy)

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("pest-2bk0e").project("detection-d0qov")
version = project.version(1)
dataset = version.download("yolov8", location="/content/raw_dataset")
print(dataset.location)

## Filter down to the same 15 best-represented classes used locally

This mirrors `src/prepare_dataset.py` from the repo, so the model trains on the exact same class set we've been working with.

In [ ]:
import shutil
from pathlib import Path

import yaml

RAW_DIR = Path("/content/raw_dataset")
OUT_DIR = Path("/content/dataset_15")

KEEP_CLASSES = [
    "Cicadellidae", "aphids", "Miridae", "blister beetle", "mole cricket",
    "grub", "Locustoidea", "wireworm", "Unaspis yanonensis",
    "legume blister beetle", "flea beetle", "flax budworm", "Prodenia litura",
    "beet army worm", "corn borer",
]

with open(RAW_DIR / "data.yaml") as f:
    raw_names = yaml.safe_load(f)["names"]

old_to_new = {i: KEEP_CLASSES.index(n) for i, n in enumerate(raw_names) if n in KEEP_CLASSES}
assert len(old_to_new) == len(KEEP_CLASSES)

kept, dropped = 0, 0
for split in ["train", "valid", "test"]:
    src_images, src_labels = RAW_DIR / split / "images", RAW_DIR / split / "labels"
    dst_images, dst_labels = OUT_DIR / split / "images", OUT_DIR / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    for label_path in src_labels.glob("*.txt"):
        lines = []
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            old_idx = int(parts[0])
            if old_idx in old_to_new:
                parts[0] = str(old_to_new[old_idx])
                lines.append(" ".join(parts))
        if not lines:
            dropped += 1
            continue
        (dst_labels / label_path.name).write_text("\n".join(lines) + "\n")
        matches = list(src_images.glob(f"{label_path.stem}.*"))
        if matches:
            shutil.copy(matches[0], dst_images / matches[0].name)
            kept += 1

out_meta = {
    "names": KEEP_CLASSES,
    "nc": len(KEEP_CLASSES),
    "train": "../train/images",
    "val": "../valid/images",
    "test": "../test/images",
}
with open(OUT_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(out_meta, f, sort_keys=False)

print(f"Kept {kept} images, dropped {dropped}")

## Train YOLOv9

`yolov9c.pt` is the compact variant (fast to fine-tune, good baseline). `patience=15` stops early if validation accuracy plateaus, so we don't burn Colab GPU time for nothing.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov9c.pt")
results = model.train(
    data=str(OUT_DIR / "data.yaml"),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,
    project="runs",
    name="pest_yolov9",
)

## Evaluate on the held-out test set (this is our real, honest accuracy number)

In [ ]:
metrics = model.val(data=str(OUT_DIR / "data.yaml"), split="test")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

## Download the trained weights back to your computer

Save the downloaded `pest_yolov9_best.pt` into the `models/` folder of the local `agri-master` project.

In [ ]:
import shutil

shutil.copy("runs/pest_yolov9/weights/best.pt", "/content/pest_yolov9_best.pt")

try:
    from google.colab import files
    files.download("/content/pest_yolov9_best.pt")
except Exception:
    print("Not running in Colab — file saved at /content/pest_yolov9_best.pt")